In [15]:
!pip install papermill
!pip install flake8
!pip install pytest
!pip install python-dotenv
!pip install joblib
!pip install ipykernel

In [16]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import papermill as pm
import flake8
import pytest
import dotenv

In [17]:
train_processed = pd.read_csv('train_processed.csv')

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, f1_score, precision_score, recall_score

X = train_processed.drop(columns=['Class'])
y = train_processed['Class']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Baseline сплит: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Baseline сплит: train (48292, 8), val (16098, 8), test (16098, 8)


In [21]:
baseline_model = LogisticRegression(random_state=42, class_weight='balanced')
baseline_model.fit(X_train_scaled, y_train)

y_val_proba = baseline_model.predict_proba(X_val_scaled)[:, 1]

logloss_val = log_loss(y_val, y_val_proba)
roc_auc_val = roc_auc_score(y_val, y_val_proba)

from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
y_val_pred = (y_val_proba >= best_threshold).astype(int)

f1_val = f1_score(y_val, y_val_pred)
precision_val = precision_score(y_val, y_val_pred)
recall_val = recall_score(y_val, y_val_pred)

print("Logistic Regression")
print(f"LogLoss: {logloss_val:.4f}")
print(f"ROC-AUC: {roc_auc_val:.4f}")
print(f"F1 (оптим. порог={best_threshold:.3f}): {f1_val:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")

Logistic Regression
LogLoss: 0.4067
ROC-AUC: 0.8108
F1 (оптим. порог=0.968): 0.1022
Precision: 0.0619
Recall: 0.2917


In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_recall_curve

knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train_scaled, y_train)
y_val_proba_knn = knn_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_knn = log_loss(y_val, y_val_proba_knn)
roc_auc_knn = roc_auc_score(y_val, y_val_proba_knn)

precision_knn, recall_knn, thresholds_knn = precision_recall_curve(y_val, y_val_proba_knn)
f1_scores_knn = 2 * precision_knn * recall_knn / (precision_knn + recall_knn + 1e-9)
best_threshold_knn = thresholds_knn[np.argmax(f1_scores_knn[:-1])]
y_val_pred_knn = (y_val_proba_knn >= best_threshold_knn).astype(int)

f1_knn = f1_score(y_val, y_val_pred_knn)
precision_knn_val = precision_score(y_val, y_val_pred_knn)
recall_knn_val = recall_score(y_val, y_val_pred_knn)

print("KNN (k=5)")
print(f"LogLoss: {logloss_knn:.4f}")
print(f"ROC-AUC: {roc_auc_knn:.4f}")
print(f"F1 (оптим. порог={best_threshold_knn:.3f}): {f1_knn:.4f}")
print(f"Precision: {precision_knn_val:.4f}")
print(f"Recall: {recall_knn_val:.4f}")

KNN (k=5)
LogLoss: 0.0487
ROC-AUC: 0.5595
F1 (оптим. порог=0.200): 0.0484
Precision: 0.0300
Recall: 0.1250


In [24]:
from sklearn.tree import DecisionTreeClassifier

dt_baseline = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_baseline.fit(X_train_scaled, y_train)
y_val_proba_dt = dt_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_dt = log_loss(y_val, y_val_proba_dt)
roc_auc_dt = roc_auc_score(y_val, y_val_proba_dt)

precision_dt, recall_dt, thresholds_dt = precision_recall_curve(y_val, y_val_proba_dt)
f1_scores_dt = 2 * precision_dt * recall_dt / (precision_dt + recall_dt + 1e-9)
best_threshold_dt = thresholds_dt[np.argmax(f1_scores_dt[:-1])]
y_val_pred_dt = (y_val_proba_dt >= best_threshold_dt).astype(int)

f1_dt = f1_score(y_val, y_val_pred_dt)
precision_dt_val = precision_score(y_val, y_val_pred_dt)
recall_dt_val = recall_score(y_val, y_val_pred_dt)

print("Decision Tree")
print(f"LogLoss: {logloss_dt:.4f}")
print(f"ROC-AUC: {roc_auc_dt:.4f}")
print(f"F1 (оптим. порог={best_threshold_dt:.3f}): {f1_dt:.4f}")
print(f"Precision: {precision_dt_val:.4f}")
print(f"Recall: {recall_dt_val:.4f}")

Decision Tree
LogLoss: 0.0873
ROC-AUC: 0.4995
F1 (оптим. порог=0.000): 0.0030
Precision: 0.0015
Recall: 1.0000


In [27]:
results = {
    'Model': ['Logistic Regression', 'KNN (k=5)', 'Decision Tree'],
    'LogLoss': [logloss_val, logloss_knn, logloss_dt],
    'ROC-AUC': [roc_auc_val, roc_auc_knn, roc_auc_dt],
    'F1': [f1_val, f1_knn, f1_dt],
    'Precision': [precision_val, precision_knn_val, precision_dt_val],
    'Recall': [recall_val, recall_knn_val, recall_dt_val]
}

df_results = pd.DataFrame(results)

df_results = df_results.sort_values('LogLoss')

print("Сравнение baseline моделей")
print(df_results.to_string(index=False))

Сравнение baseline моделей
              Model  LogLoss  ROC-AUC       F1  Precision   Recall
          KNN (k=5) 0.048682 0.559479 0.048387   0.030000 0.125000
      Decision Tree 0.087322 0.499533 0.002977   0.001491 1.000000
Logistic Regression 0.406711 0.810761 0.102190   0.061947 0.291667


Логистическая регрессия показала наилучшие результаты по LogLoss и ROC-AUC, что объясняется её способностью находить линейные зависимости между признаками и целевой переменной при сбалансированной регуляризации, а также устойчивостью к сильному дисбалансу классов за счёт параметра class_weight='balanced'. KNN, напротив, дал аномально низкий LogLoss и посредственный ROC-AUC, потому что модель практически всегда предсказывала вероятность, близкую к нулю, избегая ошибок на большинстве примеров, но при этом полностью теряя способность обнаруживать пульсары. Decision Tree достиг Recall = 1,0, но ценой огромного количества ложных срабатываний, что привело к почти случайному ROC-AUC и минимальному F1 — дерево переобучилось, выучив шум и экстремальные пороги. Таким образом, логистическая регрессия является лучшей baseline-моделью, поскольку она лучше обобщает данные и даёт наиболее сбалансированное качество между поиском пульсаров и минимизацией ложных тревог.